# RecycleNet — 6-Class Waste Classifier (Multi-Dataset)
**Datasets:** Garbage Classification + TrashNet  
**Classes:** cardboard / glass / metal / paper / plastic / trash

> ⚠️ Before running: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── CELL 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 2: Verify existing dataset ─────────────────────────────────────────
import os
DATA_DIR = '/content/drive/MyDrive/trashnet/archive/Garbage classification/Garbage classification'
print('Existing dataset:')
for cls in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(path):
        print(f'  {cls:12s}: {len(os.listdir(path))} images')

In [ ]:
# ── CELL 3: Upload kaggle.json and download extra TrashNet dataset ───────────
# Your kaggle.json is usually at: C:\Users\YourName\.kaggle\kaggle.json
from google.colab import files
import os

print('Upload your kaggle.json file:')
uploaded = files.upload()   # select kaggle.json from your computer

os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/kaggle.json')
os.system('chmod 600 /root/.kaggle/kaggle.json')

!pip install -q kaggle
!kaggle datasets download -d asdasdasasdas/garbage-classification \
    --unzip -p /content/trashnet_extra

print('Downloaded:', os.listdir('/content/trashnet_extra'))

In [ ]:
# ── CELL 4: Merge extra TrashNet images into your existing Drive dataset ─────
import os, shutil

# Source: newly downloaded TrashNet
SRC_BASE = '/content/trashnet_extra/Garbage classification/Garbage classification'

# Destination: your existing dataset on Drive
DST_BASE = '/content/drive/MyDrive/trashnet/archive/Garbage classification/Garbage classification'

classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
for cls in classes:
    src = os.path.join(SRC_BASE, cls)
    dst = os.path.join(DST_BASE, cls)
    if not os.path.exists(src):
        print(f'⚠️  Skipping {cls} — not found in downloaded dataset')
        continue
    os.makedirs(dst, exist_ok=True)
    imgs = [f for f in os.listdir(src) if not f.startswith('extra_')]
    copied = 0
    for i, img in enumerate(imgs):
        new_name = f'extra_{i}_{img}'   # prefix to avoid filename conflicts
        dest_file = os.path.join(dst, new_name)
        if not os.path.exists(dest_file):  # skip if already merged before
            shutil.copy(os.path.join(src, img), dest_file)
            copied += 1
    print(f'✅ {cls:12s}: +{copied} new → total {len(os.listdir(dst))} images')

In [ ]:
# ── CELL 5: Install dependencies ────────────────────────────────────────────
!pip install -q tensorflow scikit-learn
import tensorflow as tf, numpy as np, json
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import class_weight
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── CELL 6: Config ───────────────────────────────────────────────────────────
DATA_DIR  = '/content/drive/MyDrive/trashnet/archive/Garbage classification/Garbage classification'
IMG_SIZE  = (380, 380)
BATCH     = 32
EPOCHS_1  = 10
EPOCHS_2  = 15
NUM_CLASS = 6

In [ ]:
# ── CELL 7: Data generators with augmentation ────────────────────────────────
train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=40,
    width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.25,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True, validation_split=0.2
)
val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_ds = train_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    subset='training', shuffle=True, class_mode='sparse'
)
val_ds = val_gen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    subset='validation', shuffle=False, class_mode='sparse'
)
print('Classes:', train_ds.class_indices)
print(f'Train: {train_ds.samples} | Val: {val_ds.samples}')

In [ ]:
# ── CELL 8: Class weights ────────────────────────────────────────────────────
labels = train_ds.classes
weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(weights))
for name, idx in train_ds.class_indices.items():
    print(f'  {name:12s}: weight = {class_weights[idx]:.2f}')

In [ ]:
# ── CELL 9: Build EfficientNetB4 model ───────────────────────────────────────
base_model = EfficientNetB4(include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(NUM_CLASS, activation='softmax')(x)
model = Model(base_model.input, output)

In [ ]:
# ── CELL 10: Phase 1 — train head only, base frozen (~15 min) ────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_1,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    ]
)
print('Phase 1 best:', round(max(h1.history['val_accuracy'])*100, 2), '%')

In [ ]:
# ── CELL 11: Phase 1 decision check ──────────────────────────────────────────
p1 = max(h1.history['val_accuracy']) * 100
if p1 < 80:   print(f'❌ {p1:.1f}% — Check DATA_DIR or dataset structure')
elif p1 < 85: print(f'⚠️  {p1:.1f}% — Proceed but consider EPOCHS_1=15')
else:         print(f'✅ {p1:.1f}% — Proceed to Phase 2')

In [ ]:
# ── CELL 12: Phase 2 — fine-tune top 30 layers (~30 min) ─────────────────────
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_2,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
        tf.keras.callbacks.ModelCheckpoint(
            '/content/drive/MyDrive/trashnet/model_best.h5',
            monitor='val_accuracy', save_best_only=True)
    ]
)
print('Final:', round(max(h2.history['val_accuracy'])*100, 2), '%')

In [ ]:
# ── CELL 13: Final accuracy check ────────────────────────────────────────────
final = max(h2.history['val_accuracy']) * 100
if final >= 92:   print(f'🎯 {final:.1f}% — Excellent!')
elif final >= 90: print(f'✅ {final:.1f}% — Target hit! Save the model')
elif final >= 87: print(f'⚠️  {final:.1f}% — Rerun Cell 12 with EPOCHS_2=25')
else:             print(f'❌ {final:.1f}% — Change [:-30] to [:-50] in Cell 12')

In [ ]:
# ── CELL 14: Save model & class names ────────────────────────────────────────
import os, json
os.makedirs('/content/drive/MyDrive/trashnet', exist_ok=True)

model.save('/content/drive/MyDrive/trashnet/model_final.h5')
with open('/content/drive/MyDrive/trashnet/class_names.json', 'w') as f:
    json.dump(train_ds.class_indices, f, indent=2)

print('✅ Saved! Download these 2 files → place in your local ml/ folder:')
print('   model_final.h5')
print('   class_names.json')